# 01 · OpenAI API basics: the call, the response, streaming, errors and cost

After this notebook you can make a chat call from memory, read every field of the response, stream with time-to-first-token measured, use the Responses API, handle every error class the SDK raises, and put a dollar figure on a feature before you build it.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min (no key needed) · **Needs:** Python basics ([async/await](../01_python_basics/14_async_await_basics.ipynb) helps for the last section)

With a real key the notebook makes about 25 short calls to `gpt-4.1-mini`, which costs well under one cent.

## Why this matters in interviews

- "Walk me through an LLM API call" is the opening question of most GenAI rounds. They want: messages in, choices + `finish_reason` + `usage` out, and what you are billed for.
- Follow-ups test production instincts: streaming and time-to-first-token, retries and timeouts, which errors are worth retrying, and the cost of a multi-turn conversation.
- Coding rounds ("build a chat endpoint in 30 minutes") assume you can write this code without looking anything up.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `fo21` | What is an LLM API call actually made of, and what are you charged for? |
| `fo26` | What is the difference between a prompt, a completion and a conversation turn? |
| `pr15` | What is the difference between the system, user and assistant roles, and does it matter? |
| `fo16` | What are max tokens, stop sequences and the repetition penalty? |
| `in08` | What is streaming, and what does it actually improve? |
| `in25` | How do you handle rate limits properly? |
| `in26` | How should you handle an LLM timeout? |
| `in18` | How do you calculate cost per request, properly? |
| `pd36` | How do you estimate the cost of an AI feature before building it? |
| `tl08` | Why would you use async for an endpoint that calls an LLM? |

## 1. Setup: the key, the client, and where they come from

**In plain English:** the SDK needs to know *where* to send requests (`base_url`) and *who pays* (`api_key`). The key never goes in a notebook or a source file. It ends up in git history, screenshots and logs, and a leaked key gets used by other people within minutes.

The key lives in the **environment**:

- **Shell:** PowerShell `$env:OPENAI_API_KEY = "sk-..."` (this session) or `setx OPENAI_API_KEY "sk-..."` (new shells). Bash: `export OPENAI_API_KEY=sk-...`
- **`.env` file** next to `llm_setup.py` with a line `OPENAI_API_KEY=sk-...`. This repo's `.gitignore` already excludes `.env`. Check that in any project you join.

In an app without this repo's helper, the whole setup is two lines:

```python
from openai import OpenAI
client = OpenAI()   # reads OPENAI_API_KEY (and OPENAI_BASE_URL if set) from the environment
local  = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")   # same SDK, local Ollama
```

The notebooks use `llm_setup.py`, which builds that same `openai.OpenAI` client for whichever provider is available: OpenAI if the key is set, else a local Ollama server, else an offline stand-in, so every cell runs with no key and no cost.

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
import os, time, json, asyncio
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import openai
from llm_setup import OFFLINE, make_client, make_async_client

def api_message(e):
    """The provider's own error text (e.message also carries an 'Error code: 400 - {...}' prefix)."""
    return e.body.get("message", str(e)) if isinstance(e.body, dict) else str(e)

print(f"openai SDK version : {openai.__version__}")
print(f"OPENAI_API_KEY set : {bool(os.environ.get('OPENAI_API_KEY'))}   (never print the key itself)")
print(f"provider / model   : {PROVIDER} / {MODEL}")
print(f"client.base_url    : {client.base_url}")
print(f"client type        : {type(client).__module__}.{type(client).__name__}")

If the provider says `offline`, the answers come from a deterministic stand-in: the plumbing is real (it is the real SDK and the real response objects) but the text is canned. Where that matters, the markdown tells you what a real model returns.

## 2. Your first call, and what comes back

A Chat Completions request is: a **model id**, a list of **messages**, and optional **parameters**. The response is a `ChatCompletion` object.

In [ ]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "What is 17*23?"},
    ],
)
answer = resp.choices[0].message.content
print(answer)
assert "391" in answer

In [ ]:
choice = resp.choices[0]
usage = resp.usage
print("id             :", resp.id)
print("model          :", resp.model)
print("len(choices)   :", len(resp.choices), "(one per sample; see n= below)")
print("message.role   :", choice.message.role)
print("message.content:", choice.message.content)
print("finish_reason  :", choice.finish_reason)
print(f"usage          : {usage.prompt_tokens} prompt + {usage.completion_tokens} completion = {usage.total_tokens} tokens")
details = usage.prompt_tokens_details
print("cached prompt  :", details.cached_tokens if details else "n/a", "tokens (prompt caching, notebook 06)")
assert usage.total_tokens == usage.prompt_tokens + usage.completion_tokens

The object is a pydantic model, so `model_dump()` / `model_dump_json()` give you a plain dict or JSON for logging:

In [ ]:
print(resp.model_dump_json(indent=1, exclude_none=True)[:900])

**What you pay for:** `prompt_tokens` (everything you sent: system prompt, history, tool schemas, retrieved documents) at the input price, plus `completion_tokens` at the output price, which is usually 4-8× higher. Cached prompt tokens get a discount. Reasoning models also bill hidden **reasoning tokens** as output (`usage.completion_tokens_details.reasoning_tokens`).

**`finish_reason` is the field people forget to check:**

| `finish_reason` | Meaning | What your code should do |
|---|---|---|
| `stop` | Natural end, or a stop sequence was hit | Normal path |
| `length` | Hit the token cap (or the context limit) | The answer is **truncated**. Never parse it as JSON; raise the cap or ask for less |
| `tool_calls` | The model wants you to run tools | Execute them and send results back ([notebook 04](04_function_calling.ipynb)) |
| `content_filter` | Output was blocked by the provider's filter | Treat like a refusal; show a safe message |

## 3. Roles: who is speaking

| Role | Written by | Put here | Don't put here |
|---|---|---|---|
| `system` | You (the developer) | Job, rules, output format, tone, anything identical on every request | Per-request data or untrusted text |
| `developer` | You | OpenAI's newer name for `system` on its current models (the Responses API's `instructions=` fills the same slot) | Same as system. Many OpenAI-compatible servers only know `system`, so `system` is the portable choice |
| `user` | The end user (or your code, on their behalf) | The request and the data it is about | Your instructions mixed into their text |
| `assistant` | The model | Its previous replies, which you resend. Also hand-written example replies for few-shot | Rewrites of what you *wish* it had said |
| `tool` | Your code | Results of tool calls, matched by `tool_call_id` | Anything that does not answer a tool call (it is a 400) |

The system prompt is a strong prior, **not a security boundary**. Anything that *must* hold (permissions, spend limits) is enforced in your code.

In [ ]:
review = "The checkout was fast and the staff were friendly."
system = "Classify the sentiment of the user's message. Reply with one of: positive, negative, neutral."

with_system = client.chat.completions.create(model=MODEL, messages=[
    {"role": "system", "content": system},
    {"role": "user", "content": review}])
without_system = client.chat.completions.create(model=MODEL, messages=[
    {"role": "user", "content": review}])

label = with_system.choices[0].message.content.strip().lower().rstrip(".")
print("with a system prompt   :", label)
print("without a system prompt:", without_system.choices[0].message.content[:110])
assert label == "positive"

Same user text, two different jobs. With the system prompt, the reply is a single parseable label. Without it, a real model chats back ("Glad to hear it went well!") and the offline stand-in returns its canned placeholder. Neither of those can go into a label column.

## 4. Multi-turn: the API is stateless, so you resend the list

**In plain English:** the model does not remember your previous call. A "conversation" is just your code keeping a list and sending all of it every time. That is why the prompt gets longer (and more expensive) every turn.

In [ ]:
history = [{"role": "system", "content": "You are a friendly assistant."}]
turn_usage = []

def say(text):
    history.append({"role": "user", "content": text})
    r = client.chat.completions.create(model=MODEL, messages=history)
    reply = r.choices[0].message.content
    history.append({"role": "assistant", "content": reply})
    turn_usage.append(r.usage)
    print(f"user: {text}\n  assistant: {reply[:90]}\n  -> sent {len(history) - 1} messages = {r.usage.prompt_tokens} prompt tokens")
    return reply

say("Hi, my name is Mamitha. I work on search.")
say("Quick one: what is 12*12?")
last = say("What is my name?")
assert "Mamitha" in last

In [ ]:
fresh = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "What is my name?"}])
print("same question, no history:", fresh.choices[0].message.content)
assert "Mamitha" not in fresh.choices[0].message.content

prompt_sizes = [u.prompt_tokens for u in turn_usage]
print("prompt tokens per turn   :", prompt_sizes)
assert prompt_sizes == sorted(prompt_sizes) and prompt_sizes[-1] > prompt_sizes[0]

Two facts to say out loud in an interview:

1. **Memory is your job.** Without the history, the model has no idea who Mamitha is.
2. **Turn *n* resends turns 1…n-1**, so input tokens grow every turn and the total cost of a conversation grows roughly with the *square* of its length. [Notebook 06](06_conversation_memory_and_context_windows.ipynb) is about taming that.

## 5. The parameters that matter

| Parameter | What it does | Sensible values | Gotchas |
|---|---|---|---|
| `temperature` | Flattens (>1) or sharpens (<1) the next-token distribution before sampling | 0-0.3 for extraction/classification, 0.7-1.0 for creative text | GPT-5-family reasoning models **reject** it. `0` is near-deterministic, not guaranteed |
| `top_p` | Samples only from the smallest set of tokens whose probabilities sum to `p` | leave at 1 and tune temperature (tune one, not both) | also rejected by reasoning models |
| `max_completion_tokens` | Hard cap on generated tokens, including hidden reasoning tokens | always set one: it is your runaway-cost circuit breaker | cuts mid-sentence → `finish_reason="length"` |
| `max_tokens` | The legacy name of the cap | — | reasoning models reject it; but it is the only spelling Ollama's `/v1` reads |
| `n` | Number of independent completions for one prompt | 1; 3-5 for voting ([notebook 02](02_prompt_engineering.ipynb)) | output billed ×n; Ollama ignores it |
| `stop` | Up to 4 strings that end generation | `["\n\n"]`, `["Observation:"]` | the stop string is not included in the output |
| `seed` | Best-effort reproducibility | any fixed int | compare `system_fingerprint`; backend changes still alter output |
| `response_format` | `{"type": "json_object"}` or a JSON schema | see [notebook 03](03_structured_outputs_with_pydantic.ipynb) | JSON mode requires the word "JSON" in your messages |
| `stream` | Send tokens as they are generated | `True` for anything a human watches | usage only arrives if you ask (`stream_options`) |
| `reasoning_effort` | Reasoning models: how long to think | low / medium / high | more effort = more hidden output tokens billed |

In [ ]:
slogan = [{"role": "user", "content": "Write a one-line slogan for a coffee shop."}]

r = client.chat.completions.create(model=MODEL, messages=slogan, n=3, temperature=1.0)
samples = [c.message.content for c in r.choices]
for i, s in enumerate(samples):
    print(f"sample {i}: {s[:80]}")
print(f"asked for n=3, got {len(samples)} choice(s); distinct texts: {len(set(samples))}")
assert 1 <= len(samples) <= 3

With a real model at `temperature=1.0` the three slogans differ: that is sampling. The offline stand-in is deterministic, so its three samples are identical. If you are on Ollama you get one choice, because Ollama ignores `n`.

The cap has two spellings. Try both and see which ones the current provider honours:

In [ ]:
finish = {}
for param in ("max_completion_tokens", "max_tokens"):
    try:
        r = client.chat.completions.create(model=MODEL, messages=slogan, **{param: 5})
        finish[param] = r.choices[0].finish_reason
        print(f"{param:>21}=5 -> finish_reason={finish[param]!r:9} completion_tokens={r.usage.completion_tokens}  text={r.choices[0].message.content!r}")
    except openai.BadRequestError as e:
        finish[param] = "rejected"
        print(f"{param:>21}=5 -> rejected by {MODEL}: {api_message(e)[:80]}")
assert "length" in finish.values(), "at least one spelling of the cap should truncate the reply"

On OpenAI `gpt-4.1-mini` and offline, both spellings truncate (`finish_reason='length'`). A reasoning model (e.g. `gpt-5-mini`) rejects `max_tokens`. **Ollama silently ignores `max_completion_tokens`** and happily writes a 1,000-token story. That is a real portability trap: a cap that is silently ignored does not protect you.

**Stop sequences** end generation the moment the model writes one of the strings:

In [ ]:
r = client.chat.completions.create(
    model=MODEL, stop=["6"],
    messages=[{"role": "user", "content": "Count from 1 to 10, separated by commas. Output only the numbers."}])
text = r.choices[0].message.content
print(repr(text[:100]), "| finish_reason:", r.choices[0].finish_reason)
assert "6" not in text

A real model returns something like `'1, 2, 3, 4, 5, '`: it stops just before the `6`, and the stop string itself is never included. (The offline stand-in does not implement `stop`, so you just see its placeholder.) Agents use exactly this trick: stopping at `"Observation:"` stops a text-format ReAct agent from inventing its own tool results.

**`seed`** asks for reproducible sampling. It is best effort, so log `system_fingerprint` too: if the fingerprint changes, the backend changed and the output may too.

In [ ]:
same = [client.chat.completions.create(model=MODEL, messages=slogan, temperature=0.7, seed=42) for _ in range(2)]
for r in same:
    print(f"fingerprint={r.system_fingerprint}  text={r.choices[0].message.content[:70]!r}")
print("identical:", same[0].choices[0].message.content == same[1].choices[0].message.content)

**JSON mode** guarantees *syntactically valid JSON*, not the keys you wanted (that is structured outputs, [notebook 03](03_structured_outputs_with_pydantic.ipynb)). It also has a famous precondition:

In [ ]:
try:
    client.chat.completions.create(model=MODEL, response_format={"type": "json_object"},
                                   messages=[{"role": "user", "content": "Give me three colours."}])
    print("accepted (this server does not enforce the rule)")
except openai.BadRequestError as e:
    print(f"{e.status_code}: {api_message(e)}")

r = client.chat.completions.create(model=MODEL, response_format={"type": "json_object"},
                                   messages=[{"role": "user", "content": "Give me three colours as JSON."}])
parsed = json.loads(r.choices[0].message.content)
print("parsed JSON keys:", list(parsed))

Note the keys: a real model invents its own (`colors`, `colours`, …) and the stand-in uses `answer`. Neither is a contract your code can rely on. That gap is what JSON-schema structured outputs close.

## 6. Streaming: time-to-first-token vs total time

**In plain English:** without streaming, the user stares at a spinner until the last word is written. With streaming they start reading after the *first* word arrives. The total time is the same. What changes is **perceived** latency.

The two numbers to measure:

- **TTFT (time to first token):** network + queueing + *prefill* (reading your whole prompt). It grows with prompt length.
- **Total time:** TTFT + (output tokens × time per token). It grows with answer length.

In [ ]:
paragraph = ("Streaming sends each token to the client as soon as the model produces it, instead of waiting for the "
             "whole answer. The total generation time does not change, but the user starts reading almost immediately, "
             "which is why every chat interface streams. It also lets you cancel a long answer early and save the "
             "remaining output tokens.")
messages = [{"role": "user", "content": "Summarize this in two sentences.\n\n" + paragraph}]

if OFFLINE:
    OFFLINE.token_delay = 0.02          # make the stand-in "type" like a real model (20 ms per chunk)
t0 = time.perf_counter()
arrivals, pieces, stream_usage = [], [], None
stream = client.chat.completions.create(model=MODEL, messages=messages, stream=True,
                                        stream_options={"include_usage": True})
for chunk in stream:
    if chunk.usage:                     # the final chunk carries usage and no choices
        stream_usage = chunk.usage
    if chunk.choices and chunk.choices[0].delta.content:
        arrivals.append(time.perf_counter() - t0)
        pieces.append(chunk.choices[0].delta.content)
total = time.perf_counter() - t0
if OFFLINE:
    OFFLINE.token_delay = 0.0           # reset the knob

ttft = arrivals[0]
print("".join(pieces))
print(f"\nchunks: {len(pieces)} | TTFT: {ttft * 1000:.0f} ms | total: {total * 1000:.0f} ms | "
      f"TTFT is {ttft / total:.0%} of the wait")
print("usage from the last chunk:", stream_usage.completion_tokens if stream_usage else None, "completion tokens")
assert ttft < total and stream_usage is not None

In [ ]:
chars = np.cumsum([len(p) for p in pieces])
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.step(np.array(arrivals) * 1000, chars, where="post", lw=2, label="characters on screen (streaming)")
ax.axvline(ttft * 1000, color="tab:green", ls="--", label=f"first token: {ttft * 1000:.0f} ms")
ax.axvline(total * 1000, color="tab:red", ls="--", label=f"last token: {total * 1000:.0f} ms (non-streaming users wait this long)")
ax.set_xlabel("time since request (ms)")
ax.set_ylabel("characters received")
ax.set_title("Streaming: the reader starts at the green line instead of the red one")
ax.legend(loc="upper left", fontsize=8)
plt.show()

Production gotchas with streaming:

- `usage` only arrives in the final chunk, and only if you pass `stream_options={"include_usage": True}`. Without it, per-request cost accounting silently breaks.
- You cannot validate the whole answer before the user sees the start of it. Output guardrails either buffer (losing the benefit) or run incrementally.
- An error mid-stream arrives after you have already rendered half an answer, so your UI needs a defined "stream failed" state.
- Breaking out of the loop early and calling `stream.close()` stops generation, and you stop paying for the rest.

## 7. The Responses API: the same ideas, newer shape

OpenAI's newer endpoint. `instructions=` plays the system role, `input=` is a string or a list of messages, and `output_text` is a convenience property that joins the text output.

In [ ]:
r = client.responses.create(model=MODEL, instructions="You are a concise assistant.", input="What is 17*23?")
print("output_text :", r.output_text)
print("output items:", [item.type for item in r.output])
print(f"usage       : {r.usage.input_tokens} input + {r.usage.output_tokens} output tokens")
assert "391" in r.output_text

**Server-side conversation state.** Pass `previous_response_id` and the server prepends the earlier turns for you, so you don't have to resend the list:

In [ ]:
first = client.responses.create(model=MODEL, input="Hi, my name is Mamitha.")
second = client.responses.create(model=MODEL, input="What is my name?", previous_response_id=first.id)
print("second reply       :", second.output_text)
print("input tokens billed:", first.usage.input_tokens, "->", second.usage.input_tokens)
assert "Mamitha" in second.output_text
assert second.usage.input_tokens > first.usage.input_tokens

Look at the input tokens: **the earlier turns are still billed on every call**. Server-side state saves you bandwidth and bookkeeping, not money. Responses are stored by default (`store=True`, so the id can be referenced later). Pass `store=False` for data you must not retain, and then you are back to resending history yourself.

Streaming in the Responses API uses **typed events** instead of `chat.completion.chunk` deltas:

In [ ]:
event_counts, streamed = Counter(), []
for event in client.responses.create(model=MODEL, input="What is 12*12?", stream=True):
    event_counts[event.type] += 1
    if event.type == "response.output_text.delta":
        streamed.append(event.delta)
    elif event.type == "response.completed":
        final_usage = event.response.usage
for name, count in event_counts.items():
    print(f"{count:3d} x {name}")
print("text:", "".join(streamed), "| output tokens:", final_usage.output_tokens)
assert event_counts["response.completed"] == 1

| | Chat Completions | Responses |
|---|---|---|
| Endpoint | `POST /v1/chat/completions` | `POST /v1/responses` |
| Input | `messages=[...]` | `input="..."` or a list of items, plus `instructions=` |
| Text out | `choices[0].message.content` | `output_text` (joins the `output` items) |
| Conversation state | You resend the list | Resend, or `previous_response_id` (still billed) |
| Several samples | `n=` | not available: make several calls |
| Token cap | `max_completion_tokens` | `max_output_tokens` |
| Structured output | `response_format=` / `.parse(response_format=Model)` | `text={"format": ...}` / `.parse(text_format=Model)` |
| Tool definition | `{"type": "function", "function": {...}}` | flat: `{"type": "function", "name": ...}`, plus built-in tools (web search, file search, …) |
| Streaming | chunks with `delta` | typed events: `response.output_text.delta`, `response.completed`, … |
| Spoken by | OpenAI, Ollama, vLLM, LiteLLM, most gateways | OpenAI first; support elsewhere is growing (Ollama has it) |
| When to pick it | portability, any OpenAI-compatible server | new OpenAI-only projects, built-in tools, reasoning models |

## 8. Errors, retries and timeouts

**In plain English:** some errors mean "try again in a moment" (overloaded, rate limited, network blip) and some mean "your request is wrong" (bad parameter, bad key). Retrying the second kind just repeats the failure, more slowly and at a cost.

In [ ]:
for exc in [openai.APIConnectionError, openai.APITimeoutError, openai.BadRequestError,
            openai.AuthenticationError, openai.NotFoundError, openai.RateLimitError, openai.InternalServerError]:
    chain = " -> ".join(c.__name__ for c in exc.__mro__[:exc.__mro__.index(openai.OpenAIError) + 1])
    print(f"{exc.__name__:22} {chain}")
print("\nSDK defaults: max_retries =", openai.DEFAULT_MAX_RETRIES, "| timeout =", openai.DEFAULT_TIMEOUT)
assert issubclass(openai.APITimeoutError, openai.APIConnectionError)

| Exception | HTTP | Retry? | Typical cause |
|---|---|---|---|
| `APIConnectionError` | none (no response) | yes (the SDK does) | DNS failure, refused or reset connection, proxy |
| `APITimeoutError` | none | yes (the SDK does) | the server was slower than your `timeout` |
| `BadRequestError` | 400 | **no**: fix the request | bad parameter, invalid schema, prompt longer than the context window |
| `AuthenticationError` | 401 | **no** | missing, wrong or revoked key |
| `NotFoundError` | 404 | **no** | wrong model name (on Ollama: a model you haven't pulled) |
| `RateLimitError` | 429 | yes, with backoff | requests/tokens per minute exceeded. **Except** code `insufficient_quota`, which is a billing problem that no retry fixes |
| `InternalServerError` | ≥ 500 | yes | provider trouble |

Two things jump out of the printout. `APITimeoutError` is a **subclass** of `APIConnectionError`, so catch the timeout first. And the default timeout is **600 seconds**: no user waits ten minutes, so set your own.

The SDK already retries 408/409/429/5xx and connection errors: `max_retries=2` by default, exponential backoff from 0.5 s up to 8 s with jitter, and it honours the server's `retry-after` header. Watch it work, using the offline stand-in's fault injection:

In [ ]:
question = [{"role": "user", "content": "What is 2+2?"}]
no_retry = make_client(max_retries=0)
patient = make_client(max_retries=2)

if OFFLINE:
    OFFLINE.fail_next(1, 429)
    before = OFFLINE.request_count
    try:
        no_retry.chat.completions.create(model=MODEL, messages=question)
    except openai.RateLimitError as e:
        print(f"max_retries=0: RateLimitError {e.status_code} after {OFFLINE.request_count - before} request -> {api_message(e)}")

    OFFLINE.fail_next(2, 429)
    before, t0 = OFFLINE.request_count, time.perf_counter()
    r = patient.chat.completions.create(model=MODEL, messages=question)
    sent = OFFLINE.request_count - before
    print(f"max_retries=2: {r.choices[0].message.content!r} after {sent} requests in {time.perf_counter() - t0:.2f}s (two 429s absorbed)")
    assert sent == 3
else:
    print("Fault injection exists only in the offline stand-in; against a real API you see 429s under real load.")

In [ ]:
if OFFLINE:
    for status, exc in [(400, openai.BadRequestError), (401, openai.AuthenticationError)]:
        OFFLINE.fail_next(1, status)
        before = OFFLINE.request_count
        try:
            patient.chat.completions.create(model=MODEL, messages=question)
        except exc as e:
            sent = OFFLINE.request_count - before
            print(f"{type(e).__name__:20} ({e.status_code}) after {sent} request -> not retried, even with max_retries=2")
            assert sent == 1
else:
    print("Skipped: fault injection is offline-only.")

Timeouts and connection failures need no fault injection. A throwaway **fake server on localhost** triggers the real exceptions with no network: one that accepts connections and never answers, and one that hangs up immediately.

In [ ]:
import socket, threading
from openai import OpenAI

def fake_server(mode):
    """Local TCP server: mode='hang' never replies, mode='hangup' closes every connection at once."""
    srv = socket.socket()
    srv.bind(("127.0.0.1", 0))
    srv.listen()
    held = []
    def loop():
        while True:
            try:
                conn, _ = srv.accept()
            except OSError:
                return                      # server closed
            held.append(conn) if mode == "hang" else conn.close()
    threading.Thread(target=loop, daemon=True).start()
    return srv, f"http://127.0.0.1:{srv.getsockname()[1]}/v1"

srv, url = fake_server("hang")
slow = OpenAI(base_url=url, api_key="sk-fake", timeout=0.5, max_retries=1)
t0 = time.perf_counter()
try:
    slow.chat.completions.create(model=MODEL, messages=question)
except openai.APITimeoutError as e:
    waited = time.perf_counter() - t0
    print(f"APITimeoutError: {e}  after {waited:.1f}s = 2 attempts x 0.5s + one backoff sleep")
    assert waited >= 1.0
srv.close()

In [ ]:
srv, url = fake_server("hangup")
try:
    OpenAI(base_url=url, api_key="sk-fake", max_retries=0).chat.completions.create(model=MODEL, messages=question)
except openai.APITimeoutError:
    print("timed out (unexpected here)")
except openai.APIConnectionError as e:
    print(f"APIConnectionError: {e}  | underlying cause: {type(e.__cause__).__name__}")
srv.close()

Per-request overrides: `client.with_options(timeout=5, max_retries=0).chat.completions.create(...)`. For fine control pass an `httpx.Timeout(10.0, connect=2.0)`. Policy that holds up in production (`in25`, `in26`):

- Set the timeout **shorter than the user's patience**, not the SDK default.
- Retry only transient errors (the SDK's list), with **jitter**, honouring `retry-after`.
- Put a hard ceiling on total time, and after that fall back to a smaller model or a cached answer, marked as degraded.
- Alert on the 429 rate. A steady trickle of 429s means you need a client-side rate limiter (token bucket), not more retries.

[Retries, backoff and circuit breakers](../16_production/02_retries_backoff_rate_limits_circuit_breakers.ipynb) builds these by hand.

## 9. What does it cost?

**In plain English:** cost = input tokens × input price + output tokens × output price. The traps are the tokens you forget: the system prompt and tool schemas on *every* call, the resent history, retries, and any second model (guardrail, judge) in the request path.

In [ ]:
# USD per 1M tokens, as of Sept 2026 -- check the pricing page before quoting these.
PRICES_PER_1M = {
    "gpt-4.1-mini": {"input": 0.40, "output": 1.60},
    "gpt-4.1-nano": {"input": 0.10, "output": 0.40},
    "gpt-4o-mini":  {"input": 0.15, "output": 0.60},
    "gpt-5-mini":   {"input": 0.25, "output": 2.00},
    "gpt-5-nano":   {"input": 0.05, "output": 0.40},
    "text-embedding-3-small": {"input": 0.02, "output": 0.0},
    "text-embedding-3-large": {"input": 0.13, "output": 0.0},
}

def cost_usd(model, input_tokens, output_tokens=0):
    p = PRICES_PER_1M[model]
    return (input_tokens * p["input"] + output_tokens * p["output"]) / 1_000_000

def cost_of(usage, model="gpt-4.1-mini"):
    """Works for Chat Completions usage (prompt/completion) and Responses usage (input/output)."""
    tin = getattr(usage, "prompt_tokens", None) or getattr(usage, "input_tokens", 0)
    tout = getattr(usage, "completion_tokens", None) or getattr(usage, "output_tokens", 0)
    return cost_usd(model, tin, tout)

print(f"first call        : ${cost_of(resp.usage):.7f}")
print(f"3-turn chat       : ${sum(cost_of(u) for u in turn_usage):.7f}")
print(f"embed 1M chunks x 400 tokens: small ${cost_usd('text-embedding-3-small', 400e6):.2f} vs large ${cost_usd('text-embedding-3-large', 400e6):.2f}")
assert abs(cost_usd("gpt-4.1-mini", 1_000_000, 1_000_000) - 2.00) < 1e-9

Now the question interviewers actually ask: *"What will this feature cost per month?"* The scenario is a support bot with 20,000 requests a day, each request 1,500 input tokens (system prompt + history + retrieved docs) and 300 output tokens. There is one extra row: reasoning models also bill hidden reasoning tokens as output, so assume `gpt-5-mini` thinks for 700 tokens per answer.

In [ ]:
REQ_PER_MONTH = 20_000 * 30
scenarios = [(m, m, 1500, 300) for m in ["gpt-4.1-mini", "gpt-4.1-nano", "gpt-4o-mini", "gpt-5-mini", "gpt-5-nano"]]
scenarios.append(("gpt-5-mini\n(+700 reasoning tok)", "gpt-5-mini", 1500, 300 + 700))

labels = [s[0] for s in scenarios]
in_cost = np.array([cost_usd(m, tin) * REQ_PER_MONTH for _, m, tin, _ in scenarios])
out_cost = np.array([cost_usd(m, 0, tout) * REQ_PER_MONTH for _, m, _, tout in scenarios])

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.barh(labels, in_cost, label="input tokens")
ax.barh(labels, out_cost, left=in_cost, label="output tokens")
for i, total_cost in enumerate(in_cost + out_cost):
    ax.text(total_cost + 15, i, f"${total_cost:,.0f}", va="center", fontsize=9)
ax.invert_yaxis()
ax.set_xlim(0, (in_cost + out_cost).max() * 1.18)
ax.set_xlabel("USD per month")
ax.set_title("20k requests/day x (1,500 in + 300 out) tokens -- prices as of Sept 2026")
ax.legend(loc="upper right")
plt.show()

monthly = dict(zip(labels, in_cost + out_cost))
share_in = in_cost[0] / (in_cost[0] + out_cost[0])
print(f"gpt-4.1-mini: input is {share_in:.0%} of the bill even though output tokens cost 4x more each")
print(f"gpt-4.1-mini: ${monthly['gpt-4.1-mini']:,.0f}/month  |  gpt-5-nano: ${monthly['gpt-5-nano']:,.0f}/month")
assert round(monthly["gpt-4.1-mini"]) == 648 and share_in > 0.5

Reading the chart:

- The spread between models is ~5×. Choosing the model is the biggest cost lever, and routing easy traffic to a cheap model is the second.
- Per token, output costs 4-8× more than input. But this request sends 5× more tokens than it receives, so for the 4.1/4o models *input* is still more than half the bill. Trimming the prompt (history, retrieved chunks, tool schemas) matters as much as asking for short answers.
- Hidden reasoning tokens can move a "cheap" reasoning model up the ranking. Check `completion_tokens_details.reasoning_tokens` before you trust a per-token price.
- The real estimate also multiplies by a retry factor, adds any second model in the path, and applies the prompt-caching discount on the stable prefix (`pd36`).

## 10. The async client: many calls at once

**In plain English:** an LLM call is ~99% waiting on the network. With the async client, one Python process can have many calls in flight at once, so five calls take about as long as one. Wrap them in a semaphore so you don't blow through the provider's rate limit.

In [ ]:
aclient = make_async_client()
questions = [f"What is {a}*{b}?" for a, b in [(12, 12), (17, 23), (9, 81), (31, 7), (64, 3), (25, 4)]]
limit = asyncio.Semaphore(3)                      # at most 3 requests in flight

async def ask(q):
    async with limit:
        r = await aclient.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": q}])
        return r.choices[0].message.content

if OFFLINE:
    OFFLINE.latency = 0.3                         # pretend each request takes 300 ms, like a real API
t0 = time.perf_counter()
sequential = []
for q in questions:
    sequential.append(await ask(q))
t_seq = time.perf_counter() - t0

t0 = time.perf_counter()
concurrent = await asyncio.gather(*(ask(q) for q in questions))
t_par = time.perf_counter() - t0
if OFFLINE:
    OFFLINE.latency = 0.0

print(concurrent)
print(f"one after another: {t_seq:.2f}s | asyncio.gather: {t_par:.2f}s | speed-up x{t_seq / t_par:.1f}")
assert concurrent == sequential                   # same answers, same order
if OFFLINE:
    assert t_par < t_seq / 2
await aclient.close()

`gather` returns results **in input order**, not completion order. The speed-up only appears if the server can process requests in parallel. A hosted API can. Ollama on a CPU laptop mostly queues them (a reason it is a dev tool, not a production server). Inside FastAPI, an async endpoint must use `AsyncOpenAI`: a sync client blocks the event loop and stalls every other request (`tl08`). More in [async concurrency for LLM calls](../16_production/01_async_concurrency_for_llm_calls.ipynb).

## Try it yourself

**1. The quadratic conversation.** Each turn adds a 60-token user message and a 150-token reply, and the system prompt is 400 tokens. Write `conversation_cost(turns, model)` that sums the input *and* output cost of every call (turn *k* resends all previous turns). At which turn does the cumulative cost first exceed $0.01 on `gpt-4.1-mini`? How much does turn 40 cost compared with turn 1?

In [ ]:
# Solution — try it yourself first, then run this
def conversation_cost(turns, model="gpt-4.1-mini", system=400, user=60, reply=150):
    total, per_call = 0.0, []
    for k in range(1, turns + 1):
        prompt = system + (k - 1) * (user + reply) + user    # everything so far + the new question
        c = cost_usd(model, prompt, reply)
        per_call.append(c)
        total += c
    return total, per_call

first_over = next(t for t in range(1, 200) if conversation_cost(t)[0] > 0.01)
_, per_call = conversation_cost(40)
print(f"cumulative cost first exceeds $0.01 at turn {first_over}")
print(f"turn 1 costs ${per_call[0]:.6f}, turn 40 costs ${per_call[-1]:.6f} (x{per_call[-1] / per_call[0]:.1f})")

**2. A retry wrapper you control.** Write `robust_chat(messages, attempts=4)` that uses a client with `max_retries=0`, retries only `RateLimitError`, `APITimeoutError`, `APIConnectionError` and `InternalServerError` with *full-jitter* backoff (`sleep = random.uniform(0, base * 2**n)`), and re-raises everything else immediately. Test it with `OFFLINE.fail_next(2, 503)` (it should succeed on attempt 3) and `OFFLINE.fail_next(1, 400)` (it should fail on attempt 1).

In [ ]:
# Solution — try it yourself first, then run this
import random
random.seed(0)
RETRYABLE = (openai.RateLimitError, openai.APITimeoutError, openai.APIConnectionError, openai.InternalServerError)
bare = make_client(max_retries=0)

def robust_chat(messages, attempts=4, base=0.05):
    for n in range(attempts):
        try:
            r = bare.chat.completions.create(model=MODEL, messages=messages)
            return r.choices[0].message.content, n + 1
        except RETRYABLE:
            if n == attempts - 1:
                raise
            time.sleep(random.uniform(0, base * 2 ** n))       # full jitter

if OFFLINE:
    OFFLINE.fail_next(2, 503)
    reply, used = robust_chat(question)
    print(f"503, 503, then success: {reply!r} on attempt {used}")
    assert used == 3
    OFFLINE.fail_next(1, 400)
    try:
        robust_chat(question)
    except openai.BadRequestError as e:
        print(f"400 is not retryable -> raised immediately ({e.status_code})")
else:
    print("reply:", robust_chat(question))

**3. Cancel a stream early.** Stream a reply to "Summarize this in two sentences" (reuse `messages` from section 6) but stop after the first 40 characters. Call `stream.close()` so the server stops generating, and count how many chunks you actually received.

In [ ]:
# Solution — try it yourself first, then run this
received = ""
chunks_seen = 0
stream = client.chat.completions.create(model=MODEL, messages=messages, stream=True)
for chunk in stream:
    chunks_seen += 1
    if chunk.choices and chunk.choices[0].delta.content:
        received += chunk.choices[0].delta.content
    if len(received) >= 40:
        stream.close()                  # stop generating; you stop paying for the rest
        break
print(f"kept {len(received)} chars from {chunks_seen} chunks: {received!r}")
assert len(received) >= 40

## Say it in an interview

- **The call (30 s):** "A chat completion takes a model id, a list of role-tagged messages and sampling parameters. It returns choices, each with a message and a `finish_reason`, plus `usage`. I'm billed for prompt tokens (everything I sent, including history and tool schemas) and completion tokens, and output is typically 4-8× the input price. The API is stateless, so a conversation is my code resending the list, which makes cost grow roughly quadratically with turns."
- **Always check `finish_reason`.** `length` means truncated: never parse truncated JSON. `tool_calls` means run tools. `content_filter` means handle it as a refusal.
- **Streaming** improves *perceived* latency: TTFT is what users feel, and total time is unchanged. Ask for `include_usage` or you lose cost tracking. Plan for mid-stream errors.
- **Errors:** retry 408/409/429/5xx and connection errors with jittered exponential backoff that honours `retry-after`. Never retry 400/401/404. A 429 with `insufficient_quota` is billing, not load. The SDK does 2 retries by default, and its default timeout is 600 s, so always set your own.
- **Cost estimate:** tokens per request (system + tools + context + history + output) × price × requests × retry factor, summed over *every* model call in the path, then minus caching. Quote p50 and p99, not just the mean.
- **Traps they probe:** "Does `previous_response_id` save money?" No, the earlier turns are still billed. "Is `temperature=0` deterministic?" Nearly, not guaranteed: use `seed`, log `system_fingerprint`. "`max_tokens` or `max_completion_tokens`?" The latter on OpenAI (required for reasoning models), but OpenAI-compatible servers like Ollama may only read the former.

## Next

- [02 · Prompt engineering](02_prompt_engineering.ipynb): what to put *in* the messages, and how to measure it
- [03 · Structured outputs with pydantic](03_structured_outputs_with_pydantic.ipynb) · [04 · Function calling](04_function_calling.ipynb) · [06 · Conversation memory and context windows](06_conversation_memory_and_context_windows.ipynb)
- [Decoding: temperature, top-k, top-p](../07_genai_foundations/05_decoding_temperature_top_k_top_p.ipynb): what `temperature` does to the distribution
- [Retries, backoff, rate limits, circuit breakers](../16_production/02_retries_backoff_rate_limits_circuit_breakers.ipynb) · [Latency percentiles and cost](../16_production/05_latency_percentiles_p50_p95_and_cost.ipynb) · [FastAPI serving and streaming](../16_production/04_fastapi_serving_and_streaming.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)